In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Importações adaptadas da arquitetura do projeto
# Ajuste o import abaixo para a sua função real de conexão ao DuckDB
from painel_ocupacao_hospitalar.loaders import conectar_datasus 

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 4)

# Conexão DuckDB com as views mapeadas para os parquets de cada sistema
conn = conectar_datasus()

## 1. Filas para especialistas, exames e cirurgias
**Objetivo:** Cruzar a demanda ambulatorial/hospitalar com a infraestrutura e serviços (SIA/SIH vs CNES).
- **1.1** Ociosidade vs Sobrecarga de Equipamentos (Tomógrafos/Raios-X)
- **1.2** Taxa de Ocupação e Pressão sobre Leitos Cirúrgicos

In [ ]:
# 1.1 Ociosidade vs Sobrecarga de Equipamentos (Tomógrafos/Raios-X)
gargalos_exames = conn.execute("""
    WITH producao_exames AS (
        SELECT 
            MUNIC_RES AS cod_mun,
            COUNT(*) AS total_exames_realizados
        FROM sia
        WHERE PROC_REA LIKE '02%' -- Procedimentos com Finalidade Diagnóstica
        GROUP BY MUNIC_RES
    ),
    equipamentos_disp AS (
        SELECT 
            CODUFMUN AS cod_mun,
            SUM(CAST(QT_EXIST AS INTEGER)) AS total_equipamentos
        FROM cnes_eq
        WHERE CODEQUIP IN ('01', '02', '03') -- IDs mapeados p/ Tomógrafos, RM, Raio-X
        GROUP BY CODUFMUN
    )
    SELECT 
        i.cod_mun,
        i.nome_mun,
        i.populacao,
        COALESCE(e.total_equipamentos, 0) AS equipamentos,
        COALESCE(p.total_exames_realizados, 0) AS exames_realizados,
        ROUND(COALESCE(p.total_exames_realizados, 0) / NULLIF(e.total_equipamentos, 0), 2) AS exames_por_equipamento
    FROM ibge_pop i
    LEFT JOIN equipamentos_disp e ON i.cod_mun = e.cod_mun
    LEFT JOIN producao_exames p ON i.cod_mun = p.cod_mun
    ORDER BY exames_por_equipamento DESC
""").df()

display(gargalos_exames.head(10))

# 1.2 Taxa de Ocupação e Pressão sobre Leitos Cirúrgicos
pressao_leitos = conn.execute("""
    WITH internacoes_cirurgicas AS (
        SELECT 
            CNES,
            COUNT(*) AS volume_internacoes,
            SUM(DIAS_PERM) AS total_dias_permanencia
        FROM sih
        WHERE PROC_REA LIKE '04%' -- Procedimentos Cirúrgicos
          AND DIAS_PERM > 0
        GROUP BY CNES
    ),
    leitos_cirurgicos AS (
        SELECT 
            CNES,
            MUNICIP AS cod_mun,
            SUM(CAST(QT_EXIST AS INTEGER)) AS qtd_leitos_cirurgicos
        FROM cnes_lt
        WHERE CODLEITO IN ('33', '34') -- Cirurgia Geral / Especializada
        GROUP BY CNES, MUNICIP
    )
    SELECT 
        l.cod_mun,
        SUM(l.qtd_leitos_cirurgicos) AS total_leitos_cir,
        SUM(i.volume_internacoes) AS total_cirurgias,
        -- Estimativa de ocupação: (Dias ocupados / (Leitos * 365 dias)) * 100
        ROUND((SUM(i.total_dias_permanencia) / (SUM(l.qtd_leitos_cirurgicos) * 365.0)) * 100, 2) AS taxa_ocupacao_estimada_pct
    FROM leitos_cirurgicos l
    LEFT JOIN internacoes_cirurgicas i ON l.CNES = i.CNES
    GROUP BY l.cod_mun
    ORDER BY taxa_ocupacao_estimada_pct DESC
""").df()

display(pressao_leitos.head(10))

## 2. Desigualdade Regional de Acesso (Rotas e Fugas)
**Objetivo:** Mapear o deslocamento de pacientes e a sobrecarga de polos regionais. Avaliar municípios exportadores e polos recebedores.

In [ ]:
# 2.1 Municípios "Exportadores" de Pacientes (Fuga Assistencial)
municipios_exportadores = conn.execute("""
    SELECT 
        MUNIC_RES AS municipio_origem,
        COUNT(*) AS total_internacoes_paciente,
        SUM(CASE WHEN MUNIC_RES != MUNIC_MOV THEN 1 ELSE 0 END) AS evasao_hospitalar,
        ROUND((SUM(CASE WHEN MUNIC_RES != MUNIC_MOV THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS pct_evasao
    FROM sih
    WHERE CAST(ANO_CMPT AS INTEGER) = 2024
    GROUP BY MUNIC_RES
    HAVING COUNT(*) > 100 -- Filtro estatístico mínimo
    ORDER BY pct_evasao DESC
""").df()

display(municipios_exportadores.head())

# 2.2 Polos Regionais Sobrecarregados por Demanda Externa
polos_sobrecarregados = conn.execute("""
    SELECT 
        MUNIC_MOV AS polo_recebedor,
        COUNT(*) AS total_atendimentos,
        SUM(CASE WHEN MUNIC_RES != MUNIC_MOV THEN 1 ELSE 0 END) AS pacientes_externos,
        ROUND((SUM(CASE WHEN MUNIC_RES != MUNIC_MOV THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS pct_pressao_externa
    FROM sih
    GROUP BY MUNIC_MOV
    ORDER BY pacientes_externos DESC
""").df()

display(polos_sobrecarregados.head())

## 3. Problemas na Atenção Primária (UBS)
**Objetivo:** Avaliar a cobertura da Atenção Primária (APS) e sua correlação com Internações por Condições Sensíveis (ICSAP).

In [ ]:
# 3.1 Internações Evitáveis vs Cobertura APS
icsap_vs_aps = conn.execute("""
    WITH internacoes_csap AS (
        SELECT 
            MUNIC_RES AS cod_mun,
            COUNT(*) AS total_internacoes,
            -- Filtro simplificado de CID-10 para CSAP (ex: Asma, Hipertensão, Diabetes)
            SUM(CASE WHEN DIAG_PRINC SIMILAR TO '(J45|I10|E10|E11|E14|A09)%' THEN 1 ELSE 0 END) AS internacoes_csap
        FROM sih
        GROUP BY MUNIC_RES
    ),
    desempenho_aps AS (
        SELECT 
            CODUFMUN AS cod_mun,
            AVG(cobertura_estrategia_saude_familia) AS pct_cobertura_esf
        FROM sisab_indicadores
        GROUP BY CODUFMUN
    )
    SELECT 
        i.cod_mun,
        d.pct_cobertura_esf,
        i.total_internacoes,
        i.internacoes_csap,
        ROUND((i.internacoes_csap * 100.0) / NULLIF(i.total_internacoes, 0), 2) AS pct_internacoes_evitaveis
    FROM internacoes_csap i
    JOIN desempenho_aps d ON i.cod_mun = d.cod_mun
    ORDER BY pct_internacoes_evitaveis DESC
""").df()

display(icsap_vs_aps.head())

## 4. Falta ou Má Distribuição de Profissionais Especializados
**Objetivo:** Cruzar CBOs (Profissionais) com a infraestrutura ociosa e alta complexidade (identificar vazios assistenciais).

In [ ]:
# 4.1 Infraestrutura adequada carente de profissionais operacionais
infra_sem_profissionais = conn.execute("""
    WITH medicos_especialistas AS (
        SELECT 
            CNES,
            COUNT(DISTINCT CPF_PROF) AS qtd_anestesistas_cirurgioes
        FROM cnes_pf
        WHERE CBO IN ('225151', '225203') -- CBOs de Anestesiologista e Cirurgião Geral
        GROUP BY CNES
    ),
    equipamentos_alta_comp AS (
        SELECT 
            CNES,
            MUNICIP AS cod_mun,
            SUM(CAST(QT_EXIST AS INTEGER)) AS qtd_equip_cirurgicos
        FROM cnes_eq
        WHERE CODEQUIP IN ('05', '06') -- Ex: Mesas cirúrgicas, Monitores UTI
        GROUP BY CNES, MUNICIP
    )
    SELECT 
        eq.cod_mun,
        eq.CNES,
        eq.qtd_equip_cirurgicos,
        COALESCE(pf.qtd_anestesistas_cirurgioes, 0) AS especialistas_vinculados,
        CASE 
            WHEN COALESCE(pf.qtd_anestesistas_cirurgioes, 0) = 0 AND eq.qtd_equip_cirurgicos > 0 
            THEN 'Infra Ociosa (Falta RH)' 
            ELSE 'Operacional'
        END AS status_capacidade
    FROM equipamentos_alta_comp eq
    LEFT JOIN medicos_especialistas pf ON eq.CNES = pf.CNES
    WHERE eq.qtd_equip_cirurgicos > 0
    ORDER BY especialistas_vinculados ASC, eq.qtd_equip_cirurgicos DESC
""").df()

display(infra_sem_profissionais.head())

# 4.2 Vazios Assistenciais: Profissionais por 1.000 Habitantes
vazios_assistenciais = conn.execute("""
    WITH total_profissionais AS (
        SELECT 
            CODUFMUN AS cod_mun,
            SUM(CASE WHEN CBO LIKE '225%' THEN 1 ELSE 0 END) AS qtd_medicos,
            SUM(CASE WHEN CBO LIKE '2235%' THEN 1 ELSE 0 END) AS qtd_enfermeiros
        FROM cnes_pf
        GROUP BY CODUFMUN
    )
    SELECT 
        i.cod_mun,
        i.nome_mun,
        i.populacao,
        COALESCE(p.qtd_medicos, 0) AS medicos,
        COALESCE(p.qtd_enfermeiros, 0) AS enfermeiros,
        ROUND((COALESCE(p.qtd_medicos, 0) * 1000.0) / i.populacao, 2) AS medicos_por_mil_hab,
        ROUND((COALESCE(p.qtd_enfermeiros, 0) * 1000.0) / i.populacao, 2) AS enf_por_mil_hab
    FROM ibge_pop i
    LEFT JOIN total_profissionais p ON i.cod_mun = p.cod_mun
    WHERE i.populacao > 0
    ORDER BY medicos_por_mil_hab ASC
""").df()

display(vazios_assistenciais.head())